# Judge-Validierung: Wie gut ist der LLM-Judge wirklich?

Die Korrektheit im Modellvergleich bewerten zwei LLM-Judges (qwen2.5:14b, qwen2.5:7b); sie stimmen
untereinander nur mäßig überein (Kappa 0.39). Hier bewertest **du** 20 Antworten blind (ohne die
Judge-Urteile zu sehen). Danach vergleicht die letzte Zelle: Genauigkeit und Cohen's Kappa
Mensch ↔ Judge 14b und Mensch ↔ Judge 7b.

**Stichprobe:** 10 Antworten, die der 14b-Judge als korrekt, 10, die er als falsch wertete — über
alle Modelle gemischt, fester Seed. Gespeichert werden nur IDs + deine Labels
(`eval/judge_labels.jsonl`, kein Buchtext → darf ins Repo).

**Maßstab (wie im Judge-Prompt):** Trifft die Antwort die *Kernaussage* der Referenz? Andere
Formulierung oder zusätzliche Details sind ok; fehlt die Kernaussage oder widerspricht die Antwort
ihr, dann nicht.

In [ ]:
import json
import random
import re
from collections import Counter
from pathlib import Path

ROOT = Path("..")
ANSWERS = ROOT / "results" / "model_answers.jsonl"
VERDICTS = ROOT / "results" / "model_judge.jsonl"
GOLD = ROOT / "eval" / "gold_v2.jsonl"
LABELS_OUT = ROOT / "eval" / "judge_labels.jsonl"
JUDGES = ["qwen2.5:14b", "qwen2.5:7b-instruct-q4_k_m"]
SEED, N_PER_CLASS = 7, 10

gold = {json.loads(l)["id"]: json.loads(l) for l in GOLD.read_text().splitlines() if l.strip()}
answers = {(r["model"], r["id"]): r["answer"]
           for r in map(json.loads, ANSWERS.read_text().splitlines())}
verdicts = {v["key"]: v["ja"] for v in map(json.loads, VERDICTS.read_text().splitlines())}

# Kandidaten: vom Haupt-Judge bewertete Antworten (ohne Kalibrier-Kontrollen).
cands = []
for key, ja in verdicts.items():
    judge, model, item_id = key.split("|")
    if judge == JUDGES[0] and model not in ("cal_pos", "cal_neg"):
        cands.append({"key": f"{model}|{item_id}", "model": model, "id": item_id, "j14": ja})
rng = random.Random(SEED)
sample = (rng.sample([c for c in cands if c["j14"]], N_PER_CLASS)
          + rng.sample([c for c in cands if not c["j14"]], N_PER_CLASS))
rng.shuffle(sample)
print(f"{len(cands)} bewertete Antworten, Stichprobe {len(sample)}:",
      Counter(c["model"].split(":")[0] for c in sample))

## 1. Blind bewerten

Die Zelle zeigt jede Antwort mit Frage und Referenz — **ohne** Judge-Urteil. Trag dein Urteil in
`LABELS` ein (`"ja"` = trifft die Kernaussage, `"nein"` = nicht). Reihenfolge = Nummer.

In [ ]:
for n, c in enumerate(sample, start=1):
    item = gold[c["id"]]
    answer = re.sub(r"\s*\[\d+\]", "", answers[(c["model"], c["id"])])
    print(f"── {n:2d} ── {c['id']}")
    print(f"Frage:    {item['question']}")
    print(f"Referenz: {item['answer']}")
    print(f"Antwort:  {answer}\n")

In [ ]:
# Dein Urteil je Nummer (1..20): "ja" oder "nein"
LABELS: dict[int, str] = {
    # 1: "ja",
    # 2: "nein",
}

## 2. Speichern + Vergleich mit den Judges

In [ ]:
def kappa(a: list[bool], b: list[bool]) -> float:
    n = len(a)
    po = sum(x == y for x, y in zip(a, b)) / n
    pa, pb = sum(a) / n, sum(b) / n
    pe = pa * pb + (1 - pa) * (1 - pb)
    return (po - pe) / (1 - pe) if pe < 1 else 1.0


missing = [n for n in range(1, len(sample) + 1) if LABELS.get(n) not in ("ja", "nein")]
if missing:
    print(f"Noch offen: {missing} — erst alle 20 bewerten.")
else:
    human = [LABELS[n] == "ja" for n in range(1, len(sample) + 1)]
    rows = [{"key": c["key"], "human": h} for c, h in zip(sample, human)]
    if not LABELS_OUT.exists():  # nie überschreiben
        LABELS_OUT.write_text("".join(json.dumps(r) + "\n" for r in rows))
        print(f"gespeichert: {LABELS_OUT}")
    for judge in JUDGES:
        j = [verdicts.get(f"{judge}|{c['key']}", False) for c in sample]
        acc = sum(x == y for x, y in zip(human, j)) / len(human)
        tp = sum(h and x for h, x in zip(human, j)); tn = sum(not h and not x for h, x in zip(human, j))
        fp = sum(not h and x for h, x in zip(human, j)); fn = sum(h and not x for h, x in zip(human, j))
        print(f"{judge:<28} Genauigkeit {acc:.2f}  Kappa {kappa(human, j):.2f}   "
              f"(Mensch ja/Judge ja {tp}, nein/nein {tn}, Judge zu gnädig {fp}, zu streng {fn})")